In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 03 - Gold: Visão Brasil
# MAGIC
# MAGIC Fonte: gold.fato_alfabetizacao_municipio
# MAGIC Saída: gold.visao_brasil (Delta, partitionBy ano, ZORDER)
# MAGIC Granularidade: (ano, rede)
# MAGIC Padrão: rastreabilidade + DQ + CTAS (compatível com serverless)

# COMMAND ----------

import sys
from pathlib import Path

repo = Path.cwd()
while not repo.name.startswith("postech-aisc") and repo.parent != repo:
    repo = repo.parent
sys.path.insert(0, str(repo))

from pyspark.sql import functions as F

# COMMAND ----------

# 1. LER O FATO GOLD
df_fato = spark.table("gold.fato_alfabetizacao_municipio")

print(f"[INFO] gold.fato_alfabetizacao_municipio: {df_fato.count():,} linhas")

# COMMAND ----------

# 2. AGREGAR POR (ano, rede)
# Sem estado: visão nacional, mantendo a dimensão de rede
df_visao = (
    df_fato
    .groupBy("ano", "rede", "rede_nome")
    .agg(
        F.count("*").alias("total_municipios"),
        F.countDistinct("id_municipio").alias("municipios_distintos"),
        F.round(F.avg("resultado"), 2).alias("taxa_media"),
        F.round(F.avg("meta"), 2).alias("meta_media"),
        F.round(F.avg("folga_pp"), 2).alias("folga_media_pp"),
        F.sum(F.when(F.col("status_meta") == "ATINGIU", 1).otherwise(0)).alias("municipios_atingiram"),
        F.sum(F.when(F.col("status_meta") == "SEM_META", 1).otherwise(0)).alias("municipios_sem_meta"),
    )
    .withColumn(
        "pct_atingiram_meta",
        F.round(F.col("municipios_atingiram") / F.col("total_municipios") * 100, 2),
    )
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn("source", F.lit("gold.fato_alfabetizacao_municipio"))
    .withColumn("version", F.lit("1.0"))
)

# COMMAND ----------

# 3. DATA QUALITY
chaves = ["ano", "rede"]

total = df_visao.count()
nulos_chave = df_visao.filter(
    F.expr(" OR ".join(f"{c} IS NULL" for c in chaves))
).count()
dups = df_visao.groupBy(*chaves).count().filter(F.col("count") > 1).count()

print(f"[DQ] Total de linhas: {total}")
print(f"[DQ] Nulos na chave composta: {nulos_chave}")
print(f"[DQ] Duplicados na chave composta: {dups}")

# Persistir DQ no monitoramento
spark.sql("CREATE DATABASE IF NOT EXISTS monitoring")
spark.sql("""
  CREATE TABLE IF NOT EXISTS monitoring.dq_results (
    table_name STRING, rule STRING, status STRING,
    records_checked BIGINT, failures BIGINT, run_at TIMESTAMP
  ) USING DELTA
""")

registros = [
    ("gold.visao_brasil", "completude_chave",
     "PASS" if nulos_chave == 0 else "FAIL", int(total), int(nulos_chave)),
    ("gold.visao_brasil", "unicidade_chave_composta",
     "PASS" if dups == 0 else "FAIL", int(total), int(dups)),
]
df_dq = spark.createDataFrame(
    registros, ["table_name", "rule", "status", "records_checked", "failures"]
).withColumn("run_at", F.current_timestamp())

df_dq.createOrReplaceTempView("vw_dq_brasil")
spark.sql("INSERT INTO monitoring.dq_results SELECT * FROM vw_dq_brasil")

# COMMAND ----------

# 4. GRAVAR EM DELTA (CTAS, compatível com serverless)
spark.sql("CREATE DATABASE IF NOT EXISTS gold")

df_visao.createOrReplaceTempView("vw_visao_brasil")

spark.sql("""
CREATE OR REPLACE TABLE gold.visao_brasil
USING DELTA
PARTITIONED BY (ano)
AS SELECT * FROM vw_visao_brasil
""")

print(f"\n[OK] gold.visao_brasil gravada | Registros: {spark.table('gold.visao_brasil').count():,}")

spark.sql("OPTIMIZE gold.visao_brasil ZORDER BY (rede)")
print("[OK] ZORDER aplicado em (rede)")

# COMMAND ----------

# 5. VERIFICAÇÃO RÁPIDA
spark.sql("""
  SELECT ano, rede_nome,
         total_municipios, municipios_distintos,
         taxa_media, meta_media, pct_atingiram_meta
  FROM gold.visao_brasil
  ORDER BY ano, rede_nome
""").show()